In [ ]:
# Imputation results

import pandas as pd

df = pd.read_csv("results/imputation_results.csv")

df["method"] = pd.Categorical(df["method"], categories=["Simple", "KNN", "MissForest"], ordered=True)
df["split"] = pd.Categorical(df["split"], categories=["train", "test"], ordered=True)
df["missing_rate"] = pd.Categorical(df["missing_rate"], categories=["10%", "25%"], ordered=True)

metrics = ["rmse", "mae", "r2"]

def mean_and_empirical95CI(x: pd.Series):
    return pd.Series({
        "mean": x.mean(),
        "ci_lower_95": x.quantile(0.025),
        "ci_upper_95": x.quantile(0.975),
    })

summary = (
    df.groupby(["split", "missing_rate", "method"], observed=True)[metrics]
      .apply(lambda g: g.apply(mean_and_empirical95CI))
      .unstack(-1) 
)

summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.reset_index().sort_values(["split", "missing_rate", "method"]).reset_index(drop=True)

summary

In [1]:
# Classification results (imputed data)

import pandas as pd

df = pd.read_csv("results/classification_results.csv")
df = df[df["classifier"].isin(["LogisticRegression","XGBoost","NeuralNetwork"])]

for i in df["imputer"].unique():
    for j in sorted(df["missing_rate"].unique()):

        subset = df[
            (df["imputer"] == i) &
            (df["missing_rate"] == j)]

        if subset.empty:
            continue

        print(f"\n Imputation Method: {i} | Missing Rate: {j}")

        summary_table = (subset.groupby("classifier")[["accuracy", "f1", "roc_auc"]].mean().reset_index())
        print(summary_table)


 Imputation Method: simple | Missing Rate: 10%
           classifier  accuracy        f1   roc_auc
0  LogisticRegression  0.869733  0.869935  0.947564
1       NeuralNetwork  0.856700  0.856717  0.936595
2             XGBoost  0.834500  0.834616  0.917251

 Imputation Method: simple | Missing Rate: 25%
           classifier  accuracy        f1   roc_auc
0  LogisticRegression  0.833267  0.833677  0.916819
1       NeuralNetwork  0.813133  0.813208  0.899099
2             XGBoost  0.800367  0.800555  0.884668

 Imputation Method: knn | Missing Rate: 10%
           classifier  accuracy        f1   roc_auc
0  LogisticRegression  0.874700  0.875120  0.950017
1       NeuralNetwork  0.859933  0.860368  0.937564
2             XGBoost  0.834767  0.835176  0.916402

 Imputation Method: knn | Missing Rate: 25%
           classifier  accuracy        f1   roc_auc
0  LogisticRegression  0.832467  0.832953  0.915847
1       NeuralNetwork  0.812967  0.813776  0.897523
2             XGBoost  0.796867  0

In [ ]:
# Classification results (complete data)

import pandas as pd

df = pd.read_csv("results/classification_results_complete.csv")

df_filtered = df[df["classifier"].isin(["LogisticRegression","XGBoost","NeuralNetwork"])]

print(f"\n Complete")

summary_table = (
    df_filtered
    .groupby("classifier")[["accuracy", "f1", "roc_auc"]]
    .mean()
    .reset_index()
)

print(summary_table)